# Here's how to download the dataset from Kaggle

### These should be the first things you run to set up the finetuning process

## Kaggle Setup

In [1]:
export KAGGLE_USERNAME=$(grep KAGGLE_USERNAME .env | cut -d '=' -f2)
export KAGGLE_KEY=$(grep KAGGLE_KEY .env | cut -d '=' -f2)

echo $KAGGLE_USERNAME
echo $KAGGLE_KEY

mkdir -p ~/.config/kaggle && printf '{"username":"%s","key":"%s"}\n' "$KAGGLE_USERNAME" "$KAGGLE_KEY" > ~/.config/kaggle/kaggle.json && chmod 600 ~/.config/kaggle/kaggle.json

kaggle datasets list -s "code instruct" --sort-by votes


SyntaxError: invalid syntax (3889658058.py, line 1)

## Download the Kaggle dataset

In [ ]:
mkdir -p data/raw/evol_instruct_code_80k
kaggle datasets download -d thedevastator/evol-instruct-code-80k-v1-dataset -p data/raw/evol_instruct_code_80k

## Unzip dataset

In [ ]:
unzip data/raw/evol_instruct_code_80k/*.zip -d data/raw/evol_instruct_code_80k
ls -lh data/raw/evol_instruct_code_80k


## Check schema column names

In [ ]:
head -n 5 data/raw/evol_instruct_code_80k/train.csv


## Convert the CSV to ChatML JSONL

In [ ]:
import csv, json, os

in_path  = "data/raw/evol_instruct_code_80k/train.csv"
out_path = "data/processed/evol_code_chatml.jsonl"

n = 0
with open(in_path, "r", encoding="utf-8") as f_in, open(out_path, "w", encoding="utf-8") as f_out:
    reader = csv.DictReader(f_in)
    for row in reader:
        instr = (row.get("instruction") or "").strip()
        out   = (row.get("output") or "").strip()
        if not instr or not out:
            continue
        obj = {
            "messages": [
                {"role": "user", "content": instr},
                {"role": "assistant", "content": out},
            ]
        }
        f_out.write(json.dumps(obj, ensure_ascii=False) + "\n")
        n += 1

print(f"Wrote {n:,} rows -> {out_path}")


## Sanity check the converted file

In [ ]:
import json
path="data/processed/evol_code_chatml.jsonl"
for i,line in enumerate(open(path,"r",encoding="utf-8")):
    obj=json.loads(line)
    msgs=obj["messages"]
    print("keys:", obj.keys())
    print("roles:", [m["role"] for m in msgs])
    print("user chars:", len(msgs[0]["content"]), "assistant chars:", len(msgs[1]["content"]))
    print("user preview:", msgs[0]["content"][:120].replace("\n"," ") + "...")
    print("assistant preview:", msgs[1]["content"][:120].replace("\n"," ") + "...")
    break

## Compute token length distribution (critical before training)

In [ ]:
from transformers import AutoTokenizer
import json
from tqdm import tqdm

MODEL_ID = "openai/gpt-oss-20b"  # tokenizer only
DATA_PATH = "data/processed/evol_code_chatml.jsonl"

tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

lengths = []
for line in tqdm(open(DATA_PATH, "r", encoding="utf-8"), total=78258):
    obj = json.loads(line)
    text = tok.apply_chat_template(
        obj["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    lengths.append(len(tok(text)["input_ids"]))

lengths.sort()
def pct(p): return lengths[int(len(lengths)*p)]

print("Total samples:", len(lengths))
print("p50:", pct(0.50))
print("p90:", pct(0.90))
print("p95:", pct(0.95))
print("p99:", pct(0.99))
print("max:", lengths[-1])


# Small Pilot Training First

## Create a small pilot train config (fast “does it run?” check)

In [ ]:
# makes the evol_code_chatml_5k.jsonl dataset
import json, random, os

src = "data/processed/evol_code_chatml.jsonl"
dst = "data/processed/evol_code_chatml_5k.jsonl"
os.makedirs("data/processed", exist_ok=True)

random.seed(42)
rows = open(src, "r", encoding="utf-8").read().splitlines()
sample = random.sample(rows, 5000)

with open(dst, "w", encoding="utf-8") as f:
    for line in sample:
        f.write(line + "\n")

print("Wrote:", dst, "rows:", len(sample))


Wrote: data/processed/evol_code_chatml_5k.jsonl rows: 5000


## Run a small pilot SFT training (fast + safe) - using QWEN25coder14b for pilot run

In [3]:
from datasets import load_dataset

ds = load_dataset("json", data_files=TRAIN_JSONL, split="train").shuffle(seed=SEED)
eval_size = max(1, int(len(ds) * EVAL_RATIO))

def to_text(ex):
    return {
        "text": tokenizer.apply_chat_template(
            ex["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

def tokenize_fn(ex):
    tok = tokenizer(
        ex["text"],
        truncation=True,
        max_length=SEQ_LEN,
        padding=False,
    )
    tok["labels"] = tok["input_ids"].copy()
    return tok

# split first
raw_eval  = ds.select(range(eval_size))
raw_train = ds.select(range(eval_size, len(ds)))

# text -> tokens, and REMOVE the string columns so Trainer never sees them
ds_eval  = raw_eval.map(to_text).map(tokenize_fn, remove_columns=raw_eval.column_names + ["text"])
ds_train = raw_train.map(to_text).map(tokenize_fn, remove_columns=raw_train.column_names + ["text"])

print("train:", len(ds_train), "eval:", len(ds_eval))
print("keys:", ds_train.column_names)
print("sample lens:", len(ds_train[0]["input_ids"]))


Map: 100%|██████████| 4900/4900 [00:07<00:00, 695.18 examples/s]

train: 4900 eval: 100
keys: ['input_ids', 'attention_mask', 'labels']
sample lens: 108


In [ ]:
from transformers import TrainingArguments

# Pilot training config
targs = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=10,
    eval_strategy="steps",     # <-- your transformers version uses this
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # if this errors, use "adamw_torch"
    fp16=True,
    bf16=False,
    report_to=[],
    seed=SEED,
    remove_unused_columns=False,
    dataloader_num_workers=0,
)


In [4]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=targs,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    tokenizer=tokenizer,          # fine to keep even if deprecated
    data_collator=data_collator,
)

trainer.train()



/tmp/ipykernel_1706631/3261438146.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:467: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss,Validation Loss
50,0.418600,0.470441
100,0.421100,0.462342
150,0.415000,0.457908
200,0.421900,0.454133
250,0.413600,0.452211
300,0.429800,0.451814


/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:467: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:467: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:467: UserWarning

TrainOutput(global_step=307, training_loss=0.43427608222837166, metrics={'train_runtime': 6541.4721, 'train_samples_per_second': 0.749, 'train_steps_per_second': 0.047, 'total_flos': 1.5702786491268096e+17, 'train_loss': 0.43427608222837166, 'epoch': 1.0})